In [2]:
print("Spam Email Detection Project Started 🚀")

Spam Email Detection Project Started 🚀


In [4]:
import pandas as pd
import numpy as np
import nltk

In [5]:
import pandas as pd

df = pd.read_csv("spam.csv", encoding="latin-1")
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'spam.csv'

In [6]:
import urllib.request

# Original dataset ka direct link
url = "https://raw.githubusercontent.com/mohitgupta-omg/Kaggle-SMS-Spam-Collection-Dataset-/master/spam.csv"

print("Downloading dataset...")
# Ye line internet se file laakar tumhare folder mein 'spam.csv' naam se save karegi
urllib.request.urlretrieve(url, "spam.csv")
print("✅ Dataset downloaded successfully! Ab check karo left panel mein.")

✅ Dataset downloaded successfully! Ab check karo left panel mein.


In [7]:
import pandas as pd

# Data load karna (Kaggle data mein latin-1 encoding lagti hai)
df = pd.read_csv('spam.csv', encoding='latin-1')

# Faltu columns hatana
df = df[['v1', 'v2']] 
df.columns = ['label', 'text']

print("Data load ho gaya!")
print(df.head())

Data load ho gaya!
  label                                               text
0   ham  Go until jurong point, crazy.. Available only ...
1   ham                      Ok lar... Joking wif u oni...
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...
3   ham  U dun say so early hor... U c already then say...
4   ham  Nah I don't think he goes to usf, he lives aro...


In [8]:
import string
import nltk
from nltk.corpus import stopwords

# AI ke paas English ke faltu shabdon (stopwords) ki dictionary download kar rahe hain
nltk.download('stopwords')

def clean_message(message):
    # 1. Sab kuch lowercase karna
    message = message.lower()
    
    # 2. Punctuations (jaise !, ?, .) hatana
    message = "".join([char for char in message if char not in string.punctuation])
    
    # 3. Stopwords hatana (is, am, the)
    words = message.split()
    stop_words = set(stopwords.words('english'))
    clean_words = [word for word in words if word not in stop_words]
    
    # Wapas ek clean sentence bana kar dena
    return " ".join(clean_words)

# Apne dataframe par is safai function ko apply karna
df['clean_text'] = df['text'].apply(clean_message)

# Labels (ham/spam) ko 0 aur 1 mein convert karna (Kyunki AI text nahi, number samajhta hai)
df['label_num'] = df['label'].map({'ham': 0, 'spam': 1})

# Clean kiya hua data check karte hain
print("Text safai complete!\n")
print(df[['text', 'clean_text', 'label_num']].head())

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\abhay\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


Text safai complete!

                                                text  \
0  Go until jurong point, crazy.. Available only ...   
1                      Ok lar... Joking wif u oni...   
2  Free entry in 2 a wkly comp to win FA Cup fina...   
3  U dun say so early hor... U c already then say...   
4  Nah I don't think he goes to usf, he lives aro...   

                                          clean_text  label_num  
0  go jurong point crazy available bugis n great ...          0  
1                            ok lar joking wif u oni          0  
2  free entry 2 wkly comp win fa cup final tkts 2...          1  
3                u dun say early hor u c already say          0  
4        nah dont think goes usf lives around though          0  


In [9]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer

# Bag of Words model banana
vectorizer = CountVectorizer()

# Clean text ko matrix (numbers) mein convert karna
X = vectorizer.fit_transform(df['clean_text']) # Yeh hamara Input hai (Features)
y = df['label_num'] # Yeh hamara Output hai (Target: 0 ya 1)

# Apne data ko Training (80%) aur Testing (20%) ke liye split karna
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Total messages: {X.shape[0]}")
print("Data machine learning model ke liye ready hai!")

Total messages: 5572
Data machine learning model ke liye ready hai!


In [10]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix

# 1. Model Banana aur Train karna (AI ko sikhana)
model = MultinomialNB()
model.fit(X_train, y_train)
print("✅ Model training complete! AI ne spam ke patterns seekh liye hain.\n")

# 2. Testing Data par Predict karna (Model ka Exam)
y_pred = model.predict(X_test)

# 3. Model ka Report Card (Evaluation metrics)
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

print("--- 📊 MODEL REPORT CARD ---")
print(f"Accuracy:  {accuracy * 100:.2f}%  (Total kitne messages sahi pehchane)")
print(f"Precision: {precision * 100:.2f}%  (AI ne jisko Spam kaha, usme se asal mein kitne Spam the)")
print(f"Recall:    {recall * 100:.2f}%  (Total asal Spam messages mein se AI kitno ko pakad paaya)")

print("\n--- 🧩 CONFUSION MATRIX ---")
print(cm)
print("""
(Read as:
 [True Negatives (Sahi Ham)       False Positives (Galti se Spam bola)]
 [False Negatives (Spam miss kiya) True Positives (Sahi Spam pakda)] )
""")

✅ Model training complete! AI ne spam ke patterns seekh liye hain.

--- 📊 MODEL REPORT CARD ---
Accuracy:  97.58%  (Total kitne messages sahi pehchane)
Precision: 90.73%  (AI ne jisko Spam kaha, usme se asal mein kitne Spam the)
Recall:    91.33%  (Total asal Spam messages mein se AI kitno ko pakad paaya)

--- 🧩 CONFUSION MATRIX ---
[[951  14]
 [ 13 137]]

(Read as:
 [True Negatives (Sahi Ham)       False Positives (Galti se Spam bola)]
 [False Negatives (Spam miss kiya) True Positives (Sahi Spam pakda)] )



In [13]:
import gradio as gr

# Ek prediction function jo seedha hamare UI se connect hoga
def predict_spam(message):
    # 1. Naye text ko clean karna (apna banaya hua function use karke)
    cleaned_msg = clean_message(message)
    
    # 2. Words ko numbers (vector matrix) mein badalna
    msg_vector = vectorizer.transform([cleaned_msg])
    
    # 3. Model se predict karwana
    prediction = model.predict(msg_vector)
    
    # 4. Final Result UI par bhejna
    if prediction[0] == 1:
        return "🚨 ALERT: Yeh message SPAM (Kachra) hai!"
    else:
        return "✅ SAFE: Yeh ek normal message (Ham) hai."

# Gradio UI ka design setup
interface = gr.Interface(
    fn=predict_spam, 
    inputs=gr.Textbox(lines=4, placeholder="Koi bhi message yahan type karein (e.g., 'You won a lottery' or 'Kal meeting hai')..."),
    outputs=gr.Textbox(label="AI ki Prediction"),
    title="✉️ Spam Detector AI",
    description="Check karein ki koi email ya SMS Spam hai ya nahi. Type your custom message below!",
    theme="default"
)

# UI ko VS Code mein hi launch karna
interface.launch()

C:\Users\abhay\Downloads\Spam_Email_Detection\venv\Lib\site-packages\gradio\interface.py:171: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  super().__init__(


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Created dataset file at: .gradio\flagged\dataset1.csv


In [12]:
!python -m pip install gradio

     ---------------------------------------- 0.0/43.1 kB ? eta -:--:--
     ---------------------------------------- 43.1/43.1 kB 1.1 MB/s eta 0:00:00
     ---------------------------------------- 0.0/109.4 kB ? eta -:--:--
     -------------- ------------------------- 41.0/109.4 kB ? eta -:--:--
     -------------------------------------- 109.4/109.4 kB 2.1 MB/s eta 0:00:00
   ---------------------------------------- 0.0/20.1 MB ? eta -:--:--
    --------------------------------------- 0.3/20.1 MB 5.2 MB/s eta 0:00:04
   - -------------------------------------- 0.6/20.1 MB 8.1 MB/s eta 0:00:03
   -- ------------------------------------- 1.4/20.1 MB 9.9 MB/s eta 0:00:02
   ---- ----------------------------------- 2.0/20.1 MB 10.8 MB/s eta 0:00:02
   ----- ---------------------------------- 2.7/20.1 MB 11.6 MB/s eta 0:00:02
   ------ --------------------------------- 3.5/20.1 MB 12.4 MB/s eta 0:00:02
   ------- -------------------------------- 3.7/20.1 MB 12.6 MB/s eta 0:00:02
   -----


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [14]:
import pandas as pd

# 1. Original dataset load karna
df = pd.read_csv('spam.csv', encoding='latin-1')
print(f"Original lines: {len(df)}")

# 2. Target lines set karna
target_lines = 10000
current_lines = len(df)

if current_lines < target_lines:
    # Kitni aur lines chahiye?
    needed_lines = target_lines - current_lines
    
    # 3. Oversampling: Randomly purani lines ko duplicate karna
    sampled_df = df.sample(n=needed_lines, replace=True, random_state=42)
    
    # 4. Original aur Duplicate lines ko jodna
    new_df = pd.concat([df, sampled_df], ignore_index=True)
    
    # 5. Data ko achhe se mix (shuffle) karna taki duplicates ek sath na aayein
    new_df = new_df.sample(frac=1, random_state=42).reset_index(drop=True)
    
    # 6. Nayi file save karna
    new_df.to_csv('spam_10000.csv', index=False, encoding='latin-1')
    
    print(f"✅ Naya dataset tayyar hai! Total lines: {len(new_df)}")
    print("Tumhare folder mein 'spam_10000.csv' file save ho gayi hai.")
else:
    print("Dataset pehle se hi 10000 lines ka hai.")

Original lines: 5572
✅ Naya dataset tayyar hai! Total lines: 10000
Tumhare folder mein 'spam_10000.csv' file save ho gayi hai.


In [15]:
import joblib

# Model aur Vectorizer ko save kar rahe hain
joblib.dump(model, 'spam_model.pkl')
joblib.dump(vectorizer, 'vectorizer.pkl')

print("✅ Model aur Vectorizer save ho gaye!")

✅ Model aur Vectorizer save ho gaye!
